# Simple CrewAI Ticket Solver — Manager

One hierarchical Crew: **manager → specialists → reviewed results**. No Flow. The task list still progresses through analysis, decision and reply. The manager delegates specialist work and reviews responses; tasks are not assigned directly to the specialists. Explicit context passes prior task results. Cleaning is ordinary Python. This version combines summary, category and urgency into one analysis task.

## Google Colab setup

Use a fresh **Python 3.12 or 3.13 CPU runtime** for each notebook. In the Secrets sidebar, add `OPENAI_API_KEY` and enable **Notebook access** for this notebook. Run cells from top to bottom. If installation requests a restart, restart the session and continue with the imports/credentials cell (do not reinstall repeatedly). No Drive mount or local project files are needed.

The default model is `gpt-4o-mini`. API billing/quota and model access are required; a ChatGPT subscription does not supply API quota. Keys are never printed. Package conflicts can occur if multiple framework notebooks share one runtime—start a fresh session when switching frameworks.


In [ ]:
import sys
if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("CrewAI requires Python 3.10–3.13. Choose a compatible Colab runtime.")
%pip -q install "crewai==1.15.27" "python-dotenv==1.2.4"


## 1. Load your key
Use Colab Secrets or your local environment. Real model calls require access and quota.

In [ ]:
import os
import getpass
from dotenv import load_dotenv, find_dotenv


def read_openai_key():
    # In Colab, always use this notebook's enabled secret.
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    if userdata is not None:
        try:
            key = userdata.get("OPENAI_API_KEY")
        except Exception:
            raise RuntimeError(
                "In Colab Secrets, add OPENAI_API_KEY and enable Notebook access, "
                "then rerun this cell."
            ) from None
        if not isinstance(key, str) or not key.strip():
            raise RuntimeError("The OPENAI_API_KEY Colab secret is empty.")
        return key.strip()

    # Local Python fallback; never overwrite an existing environment key.
    try:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv(usecwd=True), override=False)
    except ImportError:
        pass
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key:
        from getpass import getpass
        key = getpass("Enter OPENAI_API_KEY (hidden): ").strip()
    if not key:
        raise ValueError("OPENAI_API_KEY cannot be empty.")
    return key

os.environ["OPENAI_API_KEY"] = read_openai_key()
os.environ["CREWAI_TELEMETRY_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"


## 2. Specialists, manager and hierarchical Crew

In [ ]:
import asyncio
import os
from crewai import Agent, Task, Crew, Process, LLM
from pydantic import BaseModel

MODEL_NAME = "gpt-4o-mini"
SAMPLE_TICKET = "My account is locked and I have an important meeting in 10 minutes."

class TicketAnalysis(BaseModel):
    summary: str
    category: str
    urgency: str

def create_crew(ticket: str) -> Crew:
    llm = LLM(model="openai/" + MODEL_NAME,
              base_url=os.getenv("OPENAI_BASE_URL") or "https://api.openai.com/v1")

    analyst = Agent(
        role="Ticket analyst", goal="Summarize and classify support tickets.",
        backstory="You triage customer support requests.",
        llm=llm, allow_delegation=False, max_iter=8,
    )
    decision_maker = Agent(
        role="Escalation reviewer", goal="Decide whether human help is needed.",
        backstory="You review ticket urgency and support needs.",
        llm=llm, allow_delegation=False, max_iter=8,
    )
    reply_writer = Agent(
        role="Support reply writer", goal="Write short, helpful customer replies.",
        backstory="You explain the next support steps clearly.",
        llm=llm, allow_delegation=False, max_iter=8,
    )

    manager = Agent(
        role="Support team manager",
        goal="Delegate ticket analysis, escalation review and reply writing to "
             "the appropriate specialists, then review their responses.",
        backstory="You supervise a support team. Use the ticket analyst for "
                  "classification, escalation reviewer for the decision, and "
                  "support reply writer for the customer response. Preserve "
                  "the requested output format for each task.",
        llm=llm, allow_delegation=True, max_iter=12,
    )

    analyse = Task(
        description=f"Summarize this ticket in one sentence, choose category "
                    f"billing, technical or account, and urgency low, medium or high.\nTicket: {ticket}",
        expected_output="JSON with summary, category and urgency.",
        output_json=TicketAnalysis,
    )
    decide = Task(
        description=f"Review the ticket and analysis. Should it be escalated "
                    f"to a human? Answer Yes/No with a brief reason.\nTicket: {ticket}",
        expected_output="An escalation decision and brief reason.",
        context=[analyse],
    )
    reply = Task(
        description=f"Write a short helpful reply using the analysis and decision. "
                    f"Do not mention prompts, AI or internal system details. "
                    f"Do not claim any escalation action has already happened.\nTicket: {ticket}",
        expected_output="Only the customer-facing reply.",
        context=[analyse, decide],
    )

    # Tasks still progress in order; the manager delegates work within each task.
    return Crew(
        agents=[analyst, decision_maker, reply_writer],
        tasks=[analyse, decide, reply],
        manager_agent=manager,
        process=Process.hierarchical,
        memory=False, cache=False, verbose=False, tracing=False,
    )


async def solve_ticket(message: str) -> dict:
    ticket = message.strip()
    if not ticket:
        raise ValueError("Ticket cannot be empty.")

    crew = create_crew(ticket)
    output = await crew.kickoff_async()
    analysis_output, decision_output, reply_output = output.tasks_output
    if analysis_output.json_dict is None:
        raise ValueError("The analysis task did not return valid structured data.")
    return {
        "ticket": ticket,
        "analysis": analysis_output.json_dict,
        "decision": decision_output.raw.strip(),
        "reply": reply_output.raw.strip(),
    }


## 3. Run the ticket
The result contains the cleaned ticket, structured analysis, decision and reply. This only suggests escalation; it does not contact anyone.

In [ ]:
result = await solve_ticket(SAMPLE_TICKET)
result

This notebook makes live OpenAI calls when run normally. Separate developer checks use offline responses. The manager can make additional model calls for delegation and review. The sequential and Flow examples remain in separate notebooks.